# Experiment 3: Text Generation using FLAN-T5 for a Question–Answer System
**Aim:** Use the pretrained `google/flan-t5-base` Transformer model to answer user questions and provide an interactive Gradio interface. No model training is needed.

In [ ]:
%pip install -q transformers accelerate sentencepiece gradio

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 1. Load pretrained FLAN-T5

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MODEL_NAME = 'google/flan-t5-base'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print('Model loaded on', DEVICE)

## 2. Define the question-answering function

In [ ]:
def answer_question(question, max_tokens=128):
    question = (question or '').strip()
    if not question:
        return 'Please enter a question.'
    prompt = f'Answer the following question clearly and concisely:\nQuestion: {question}\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=int(max_tokens), num_beams=4, early_stopping=True)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(answer_question('What is artificial intelligence?'))

## 3. Create the Gradio interface

In [ ]:
import gradio as gr
demo = gr.Interface(
    fn=answer_question,
    inputs=[gr.Textbox(label='Enter your question', placeholder='What is artificial intelligence?', lines=3),
            gr.Slider(32, 256, value=128, step=16, label='Maximum answer tokens')],
    outputs=gr.Textbox(label='Generated answer', lines=6),
    title='FLAN-T5 Question–Answer System',
    description='Ask a question and generate an answer using a pretrained Transformer model.',
    examples=[['What is machine learning?', 128], ['Explain attention in simple words.', 128]]
)
demo.launch(share=True)

## 4. Test different questions
Try changing the question in the Gradio interface. Save a screenshot showing a question and its generated answer for your lab record.